# =========================================================
# Feature engineering for df_features
# =========================================================

In [71]:
# =========================================================
# 1. Load the prepared master dataset for feature engineering
# =========================================================

# Import pandas for tabular data handling
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Load the prepared master dataset and parse the timestamp column as datetime
df = pd.read_csv("../data/df_final.csv", parse_dates=["timestamp"])

# Sort the dataset chronologically and reset the row index
df = df.sort_values("timestamp").reset_index(drop=True)

# Display the shape and the available columns for a first check
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (62728, 30)
Columns: ['timestamp', 'price', 'load', 'wind_offshore', 'wind_onshore', 'solar', 'hour', 'day_of_week', 'month', 'temperature', 'wind_speed', 'is_weekend', 'gas_price', 'coal_price', 'price_lag_24h', 'price_lag_168h', 'price_rolling_24h', 'price_rolling_168h', 'co2_price', 'is_holiday', 'is_hol_or_week', 'total_generation', 'net_export', 'coal_generation', 'gas_generation', 'nuclear_generation', 'actual_wind_offshore', 'actual_wind_onshore', 'actual_solar', 'actual_load']


In [72]:
# =========================================================
# 2. Recreate safe price-history features using only past prices
# =========================================================

# Create the electricity price from 24 hours earlier
df["price_lag_24h"] = df["price"].shift(24)

# Create the electricity price from 168 hours earlier (same hour one week before)
df["price_lag_168h"] = df["price"].shift(168)

# Create the rolling 24-hour average price using only past prices
# shift(1) ensures that the current hour price is not included
df["price_rolling_24h"] = df["price"].shift(1).rolling(24).mean()

# Create the rolling 168-hour average price using only past prices
# shift(1) ensures that the current hour price is not included
df["price_rolling_168h"] = df["price"].shift(1).rolling(168).mean()

# Create the rolling 24-hour price volatility using only past prices
# shift(1) ensures that the current hour price is not included
df["price_volatility_24h"] = df["price"].shift(1).rolling(24).std()

# Check the new price-history columns
print("NaNs from safe price-history features:")
print(df[[
    "price_lag_24h",
    "price_lag_168h",
    "price_rolling_24h",
    "price_rolling_168h",
    "price_volatility_24h"
]].isnull().sum())

NaNs from safe price-history features:
price_lag_24h            24
price_lag_168h          168
price_rolling_24h        24
price_rolling_168h      168
price_volatility_24h     24
dtype: int64


In [73]:
# =========================================================
# 4. Create lagged fuel-price features
# =========================================================

# Create gas price lag features from 24 hours and 168 hours earlier
df["gas_price_lag_24h"] = df["gas_price"].shift(24)
df["gas_price_lag_168h"] = df["gas_price"].shift(168)

# Create coal price lag features from 24 hours and 168 hours earlier
df["coal_price_lag_24h"] = df["coal_price"].shift(24)
df["coal_price_lag_168h"] = df["coal_price"].shift(168)

# Create CO2 price lag feature from 24 hours earlier
df["co2_price_lag_24h"] = df["co2_price"].shift(24)

# Check the expected missing values from the lag construction
print("NaNs from fuel lags:")
lag_cols = [
    "gas_price_lag_24h", "gas_price_lag_168h",
    "coal_price_lag_24h", "coal_price_lag_168h",
    "co2_price_lag_24h"
]
print(df[lag_cols].isnull().sum())

NaNs from fuel lags:
gas_price_lag_24h       24
gas_price_lag_168h     168
coal_price_lag_24h      24
coal_price_lag_168h    168
co2_price_lag_24h       24
dtype: int64


In [74]:
# =========================================================
# 4.1 Create lagged system-state features from past known values
# =========================================================

# Create lagged total generation features from 24 hours and 168 hours earlier
# These use past realized values, so they are known at prediction time
df["total_generation_lag_24h"] = df["total_generation"].shift(24)
df["total_generation_lag_168h"] = df["total_generation"].shift(168)

# Create lagged net export features from 24 hours and 168 hours earlier
# These also use past realized values, not same-hour realized values
df["net_export_lag_24h"] = df["net_export"].shift(24)
df["net_export_lag_168h"] = df["net_export"].shift(168)

# Check the expected missing values from the lag construction
print("NaNs from lagged system-state features:")
print(df[[
    "total_generation_lag_24h",
    "total_generation_lag_168h",
    "net_export_lag_24h",
    "net_export_lag_168h"
]].isnull().sum())

NaNs from lagged system-state features:
total_generation_lag_24h      24
total_generation_lag_168h    168
net_export_lag_24h            24
net_export_lag_168h          168
dtype: int64


In [75]:
# =========================================================
# 5. Create interaction features based only on forecast-safe inputs
# =========================================================

# Create a peak-hour flag for morning and evening demand peaks
df["is_peak_hour"] = df["hour"].isin(range(7, 11)) | df["hour"].isin(range(17, 22))
df["is_peak_hour"] = df["is_peak_hour"].astype(int)

# Create wind generation during peak hours
# wind_onshore and wind_offshore are day-ahead forecast values in this dataset
df["wind_x_peak"] = (df["wind_onshore"] + df["wind_offshore"]) * df["is_peak_hour"]

# Create gas price pressure during peak hours
df["gas_x_peak"] = df["gas_price"] * df["is_peak_hour"]

# Create solar generation during higher-demand hours
# Both solar and load are forecast-based columns in this dataset
df["solar_x_demand"] = df["solar"] * df["load"]

# Check the interaction features
interact_cols = ["is_peak_hour", "wind_x_peak", "gas_x_peak", "solar_x_demand"]
print(df[interact_cols].describe().round(2))
print(f"\nPeak hours: {df['is_peak_hour'].sum():,} rows ({df['is_peak_hour'].mean()*100:.1f}%)")

       is_peak_hour  wind_x_peak  gas_x_peak  solar_x_demand
count      62728.00     62728.00    62728.00    6.272800e+04
mean           0.38      5530.37       16.84    3.611918e+08
std            0.48      9768.79       34.72    5.625654e+08
min            0.00         0.00        0.00    0.000000e+00
25%            0.00         0.00        0.00    0.000000e+00
50%            0.00         0.00        0.00    1.151225e+07
75%            1.00      7630.31       24.76    5.707494e+08
max            1.00     51550.50      339.20    3.067909e+09

Peak hours: 23,526 rows (37.5%)


In [76]:
# =========================================================
# 6. Create additional safe time-regime and forecast-based features
# =========================================================

# Mark the energy crisis period from 2021 to 2023
df["is_crisis_period"] = (
    (df["timestamp"] >= "2021-01-01") &
    (df["timestamp"] <= "2023-12-31")
).astype(int)

# Create a negative price flag for diagnostics only
# This column is derived from the target and will not be used as a model feature
df["is_negative_price"] = (df["price"] < 0).astype(int)

# Keep the calendar year as a long-term trend feature
df["year"] = df["timestamp"].dt.year

# Create residual load from forecasted load and forecasted renewables
df["residual_load"] = df["load"] - (
    df["wind_onshore"] + df["wind_offshore"] + df["solar"]
)

# Create the hour-to-hour change in forecasted load
df["load_ramp"] = df["load"].diff(1)

# Create the total forecasted renewable generation
renew_forecast = df["wind_onshore"] + df["wind_offshore"] + df["solar"]

# Create the hour-to-hour change in forecasted renewable generation
df["renewable_ramp"] = renew_forecast.diff(1)

# Create the total wind forecast
df["total_wind_forecast"] = df["wind_onshore"] + df["wind_offshore"]

# Create the change in total wind forecast compared with 24 hours earlier
df["delta_wind_forecast"] = df["total_wind_forecast"] - df["total_wind_forecast"].shift(24)

# Check the newly created features
print(f"Crisis period rows:    {df['is_crisis_period'].sum():,} ({df['is_crisis_period'].mean()*100:.1f}%)")
print(f"Negative price rows:   {df['is_negative_price'].sum():,} ({df['is_negative_price'].mean()*100:.1f}%)")
print(f"Years in dataset:      {sorted(df['year'].unique())}")

print("\nNaNs in additional engineered features:")
print(df[[
    "residual_load",
    "load_ramp",
    "renewable_ramp",
    "price_volatility_24h",
    "total_wind_forecast",
    "delta_wind_forecast"
]].isnull().sum())

Crisis period rows:    26,254 (41.9%)
Negative price rows:   2,037 (3.2%)
Years in dataset:      [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

NaNs in additional engineered features:
residual_load            0
load_ramp                1
renewable_ramp           1
price_volatility_24h    24
total_wind_forecast      0
delta_wind_forecast     24
dtype: int64


In [77]:
# =========================================================
# 7. Check missing values before removing incomplete feature rows
# =========================================================

# Identify all columns that currently contain missing values
nan_cols = df.columns[df.isnull().any()].tolist()

# Display the number of missing values in those columns
print("NaNs per column before drop:")
print(df[nan_cols].isnull().sum().sort_values(ascending=False))

NaNs per column before drop:
price_lag_168h               168
net_export_lag_168h          168
price_rolling_168h           168
coal_price_lag_168h          168
gas_price_lag_168h           168
total_generation_lag_168h    168
price_lag_24h                 24
price_volatility_24h          24
price_rolling_24h             24
coal_price_lag_24h            24
gas_price_lag_24h             24
total_generation_lag_24h      24
co2_price_lag_24h             24
net_export_lag_24h            24
delta_wind_forecast           24
load_ramp                      1
renewable_ramp                 1
dtype: int64


In [78]:
# =========================================================
# 8. Drop rows where required model features are not yet available
# =========================================================

# Define the engineered columns that require past values and therefore create initial NaNs
drop_cols = [
    "price_lag_24h",
    "price_lag_168h",
    "price_rolling_24h",
    "price_rolling_168h",
    "price_volatility_24h",
    "gas_price_lag_24h",
    "gas_price_lag_168h",
    "coal_price_lag_24h",
    "coal_price_lag_168h",
    "co2_price_lag_24h",
    "total_generation_lag_24h",
    "total_generation_lag_168h",
    "net_export_lag_24h",
    "net_export_lag_168h",
    "load_ramp",
    "renewable_ramp",
    "delta_wind_forecast"
]

# Remove rows where these required features are still missing
df = df.dropna(subset=drop_cols).reset_index(drop=True)

# Verify the result after dropping incomplete rows
print(f"Shape after drop: {df.shape}")
print(f"Remaining NaNs:   {df.isnull().sum().sum()}")
print(f"New date range:   {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

Shape after drop: (62560, 52)
Remaining NaNs:   0
New date range:   2019-01-15 → 2026-03-05


In [79]:
# =========================================================
# 9. Define the columns that must be excluded from model training
# =========================================================

# Exclude the timestamp and target column
# Also exclude diagnostic columns and realized columns that are not available at prediction time
EXCLUDE_COLS = [
    "timestamp",
    "date",
    "price",
    "is_negative_price",
    "actual_wind_offshore",
    "actual_wind_onshore",
    "actual_solar",
    "actual_load",
    "total_generation",
    "coal_generation",
    "gas_generation",
    "nuclear_generation",
    "net_export"
]

# Build the final list of model feature columns
feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

# Display a summary of the final feature selection
print(f"Total columns:   {df.shape[1]}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Excluded:        {len(EXCLUDE_COLS)}")

print("\nFeatures going into model:")
for col in feature_cols:
    print(f"  {col}")

Total columns:   52
Feature columns: 40
Excluded:        13

Features going into model:
  load
  wind_offshore
  wind_onshore
  solar
  hour
  day_of_week
  month
  temperature
  wind_speed
  is_weekend
  gas_price
  coal_price
  price_lag_24h
  price_lag_168h
  price_rolling_24h
  price_rolling_168h
  co2_price
  is_holiday
  is_hol_or_week
  price_volatility_24h
  gas_price_lag_24h
  gas_price_lag_168h
  coal_price_lag_24h
  coal_price_lag_168h
  co2_price_lag_24h
  total_generation_lag_24h
  total_generation_lag_168h
  net_export_lag_24h
  net_export_lag_168h
  is_peak_hour
  wind_x_peak
  gas_x_peak
  solar_x_demand
  is_crisis_period
  year
  residual_load
  load_ramp
  renewable_ramp
  total_wind_forecast
  delta_wind_forecast


In [80]:
# =========================================================
# 8.1 Impute missing spring-DST hours and drop the final incomplete day
# =========================================================

# Create a date column for day-based checks
df["date"] = df["timestamp"].dt.date

# Count rows per day
day_counts = df.groupby("date").size()

# Identify incomplete days
incomplete_days = day_counts[day_counts != 24].index.tolist()

print("Incomplete days before fix:")
print(day_counts[day_counts != 24])

# Separate spring DST days from the final incomplete dataset-end day
dst_days = []
other_incomplete_days = []

for d in incomplete_days:
    ts = pd.Timestamp(d)
    if ts.month == 3 and ts.day in range(25, 32):
        dst_days.append(d)
    else:
        other_incomplete_days.append(d)

print("\nSpring DST days to impute:")
print(dst_days)

print("\nOther incomplete days to drop:")
print(other_incomplete_days)

# Impute the missing 02:00 hour on each spring DST day
new_rows = []

for d in dst_days:
    missing_ts = pd.Timestamp(f"{d} 02:00:00")

    # Use the neighboring hours 01:00 and 03:00
    prev_ts = missing_ts - pd.Timedelta(hours=1)
    next_ts = missing_ts + pd.Timedelta(hours=1)

    prev_row = df[df["timestamp"] == prev_ts]
    next_row = df[df["timestamp"] == next_ts]

    # Only impute if both neighbors exist
    if len(prev_row) == 1 and len(next_row) == 1:
        prev_row = prev_row.iloc[0]
        next_row = next_row.iloc[0]

        new_row = prev_row.copy()
        new_row["timestamp"] = missing_ts
        new_row["date"] = missing_ts.date()
        new_row["hour"] = missing_ts.hour
        new_row["day_of_week"] = missing_ts.dayofweek
        new_row["month"] = missing_ts.month
        new_row["year"] = missing_ts.year
        new_row["is_weekend"] = missing_ts.dayofweek >= 5

        # Interpolate numeric columns linearly between 01:00 and 03:00
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

        # Do not overwrite binary calendar/flag columns that should be set explicitly
        protected_cols = [
            "hour", "day_of_week", "month", "year",
            "is_weekend", "is_holiday", "is_hol_or_week",
            "is_peak_hour", "is_crisis_period", "is_negative_price"
        ]

        for col in numeric_cols:
            if col not in protected_cols:
                new_row[col] = (prev_row[col] + next_row[col]) / 2

        # Recompute holiday-related flags from the timestamp
        if "is_holiday" in df.columns:
            new_row["is_holiday"] = prev_row["is_holiday"]
        if "is_hol_or_week" in df.columns:
            new_row["is_hol_or_week"] = bool(new_row["is_weekend"] or new_row.get("is_holiday", False))

        # Recompute cyclic features if they exist
        if "hour_sin" in df.columns:
            new_row["hour_sin"] = np.sin(2 * np.pi * new_row["hour"] / 24)
        if "hour_cos" in df.columns:
            new_row["hour_cos"] = np.cos(2 * np.pi * new_row["hour"] / 24)
        if "dow_sin" in df.columns:
            new_row["dow_sin"] = np.sin(2 * np.pi * new_row["day_of_week"] / 7)
        if "dow_cos" in df.columns:
            new_row["dow_cos"] = np.cos(2 * np.pi * new_row["day_of_week"] / 7)
        if "month_sin" in df.columns:
            new_row["month_sin"] = np.sin(2 * np.pi * new_row["month"] / 12)
        if "month_cos" in df.columns:
            new_row["month_cos"] = np.cos(2 * np.pi * new_row["month"] / 12)

        new_rows.append(new_row)

# Append imputed rows
if new_rows:
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

# Drop other incomplete days, such as the final truncated day
if other_incomplete_days:
    df = df[~df["date"].isin(other_incomplete_days)].copy()

# Sort again after modifications
df = df.sort_values("timestamp").reset_index(drop=True)

# Recheck day completeness
day_counts_after = df.groupby(df["timestamp"].dt.date).size()

print("\nRows per day after fix:")
print(day_counts_after.value_counts().sort_index())

print("\nRemaining incomplete days after fix:")
print(day_counts_after[day_counts_after != 24])

print("\nNew shape:")
print(df.shape)
print(f"New date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Incomplete days before fix:
date
2019-03-31    23
2020-03-29    23
2021-03-28    23
2022-03-27    23
2023-03-26    23
2024-03-31    23
2025-03-30    23
2026-03-05    23
dtype: int64

Spring DST days to impute:
[datetime.date(2019, 3, 31), datetime.date(2020, 3, 29), datetime.date(2021, 3, 28), datetime.date(2022, 3, 27), datetime.date(2023, 3, 26), datetime.date(2024, 3, 31), datetime.date(2025, 3, 30)]

Other incomplete days to drop:
[datetime.date(2026, 3, 5)]

Rows per day after fix:
24    2606
Name: count, dtype: int64

Remaining incomplete days after fix:
Series([], dtype: int64)

New shape:
(62544, 53)
New date range: 2019-01-15 00:00:00 to 2026-03-04 23:00:00


In [81]:
# =========================================================
# 9. Define the final strictly leakage-free feature set
# =========================================================

EXCLUDE_COLS = [
    "timestamp",
    "date",
    "price",
    "is_negative_price",
    "actual_wind_offshore",
    "actual_wind_onshore",
    "actual_solar",
    "actual_load",
    "total_generation",
    "coal_generation",
    "gas_generation",
    "nuclear_generation",
    "net_export",
    "gas_price",
    "coal_price",
    "co2_price",
    "gas_x_peak"
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"Total columns:   {df.shape[1]}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Excluded:        {len(EXCLUDE_COLS)}")

print("\nStrictly leakage-free features going into model:")
for col in feature_cols:
    print(f"  {col}")

Total columns:   53
Feature columns: 36
Excluded:        17

Strictly leakage-free features going into model:
  load
  wind_offshore
  wind_onshore
  solar
  hour
  day_of_week
  month
  temperature
  wind_speed
  is_weekend
  price_lag_24h
  price_lag_168h
  price_rolling_24h
  price_rolling_168h
  is_holiday
  is_hol_or_week
  price_volatility_24h
  gas_price_lag_24h
  gas_price_lag_168h
  coal_price_lag_24h
  coal_price_lag_168h
  co2_price_lag_24h
  total_generation_lag_24h
  total_generation_lag_168h
  net_export_lag_24h
  net_export_lag_168h
  is_peak_hour
  wind_x_peak
  solar_x_demand
  is_crisis_period
  year
  residual_load
  load_ramp
  renewable_ramp
  total_wind_forecast
  delta_wind_forecast


In [82]:
# =========================================================
# 10. Save the final leakage-free feature dataset for modeling
# =========================================================

save_cols = ["timestamp", "price"] + feature_cols

df_features_final = df[save_cols].copy()
df_features_final.to_csv("../data/df_features.csv", index=False)

print(f"Saved: df_features.csv  ({df_features_final.shape[0]:,} rows, {df_features_final.shape[1]} cols)")
print("Saved columns:")
print(df_features_final.columns.tolist())

Saved: df_features.csv  (62,544 rows, 38 cols)
Saved columns:
['timestamp', 'price', 'load', 'wind_offshore', 'wind_onshore', 'solar', 'hour', 'day_of_week', 'month', 'temperature', 'wind_speed', 'is_weekend', 'price_lag_24h', 'price_lag_168h', 'price_rolling_24h', 'price_rolling_168h', 'is_holiday', 'is_hol_or_week', 'price_volatility_24h', 'gas_price_lag_24h', 'gas_price_lag_168h', 'coal_price_lag_24h', 'coal_price_lag_168h', 'co2_price_lag_24h', 'total_generation_lag_24h', 'total_generation_lag_168h', 'net_export_lag_24h', 'net_export_lag_168h', 'is_peak_hour', 'wind_x_peak', 'solar_x_demand', 'is_crisis_period', 'year', 'residual_load', 'load_ramp', 'renewable_ramp', 'total_wind_forecast', 'delta_wind_forecast']


In [83]:
# =========================================================
# Final data quality check for df_features
# =========================================================

# Load the saved feature dataset
df_check = pd.read_csv("../data/df_features.csv", parse_dates=["timestamp"])

# Show basic shape and range
print("=== SHAPE ===")
print(df_check.shape)

print("\n=== DATE RANGE ===")
print(f"From: {df_check['timestamp'].min()}")
print(f"To:   {df_check['timestamp'].max()}")

# Check duplicate timestamps
print("\n=== DUPLICATE TIMESTAMPS ===")
print(df_check["timestamp"].duplicated().sum())

# Check missing values
print("\n=== MISSING VALUES PER COLUMN ===")
print(df_check.isnull().sum().sort_values(ascending=False))

print("\n=== TOTAL NUMBER OF MISSING VALUES ===")
print(df_check.isnull().sum().sum())

# Check daily completeness
print("\n=== ROWS PER DAY ===")
df_check["date"] = df_check["timestamp"].dt.date
day_counts = df_check.groupby("date").size()
print(day_counts.value_counts().sort_index())

print("\n=== INCOMPLETE OR OVERCOMPLETE DAYS ===")
display(day_counts[day_counts != 24])

# Check for time gaps
print("\n=== TIME GAP CHECK ===")
full_range = pd.date_range(
    start=df_check["timestamp"].min(),
    end=df_check["timestamp"].max(),
    freq="h"
)
missing_hours = full_range.difference(df_check["timestamp"])
print(f"Missing hours: {len(missing_hours)}")
print(missing_hours[:20])

# Columns where zero can be suspicious
suspicious_zero_cols = [
    "load", "temperature", "wind_speed",
    "gas_price_lag_24h", "gas_price_lag_168h",
    "coal_price_lag_24h", "coal_price_lag_168h",
    "co2_price_lag_24h",
    "price_lag_24h", "price_lag_168h",
    "price_rolling_24h", "price_rolling_168h",
    "price_volatility_24h",
    "total_generation_lag_24h", "total_generation_lag_168h",
    "net_export_lag_24h", "net_export_lag_168h"
]

existing_suspicious_cols = [c for c in suspicious_zero_cols if c in df_check.columns]

print("\n=== ZERO COUNTS IN SUSPICIOUS COLUMNS ===")
zero_counts = (df_check[existing_suspicious_cols] == 0).sum().sort_values(ascending=False)
print(zero_counts)

# Show descriptive statistics
print("\n=== DESCRIPTIVE STATISTICS ===")
display(df_check.describe().round(3))

# Preview
print("\n=== HEAD ===")
display(df_check.head(3))

print("\n=== TAIL ===")
display(df_check.tail(3))

=== SHAPE ===
(62544, 38)

=== DATE RANGE ===
From: 2019-01-15 00:00:00
To:   2026-03-04 23:00:00

=== DUPLICATE TIMESTAMPS ===
0

=== MISSING VALUES PER COLUMN ===
timestamp                    0
price                        0
load                         0
wind_offshore                0
wind_onshore                 0
solar                        0
hour                         0
day_of_week                  0
month                        0
temperature                  0
wind_speed                   0
is_weekend                   0
price_lag_24h                0
price_lag_168h               0
price_rolling_24h            0
price_rolling_168h           0
is_holiday                   0
is_hol_or_week               0
price_volatility_24h         0
gas_price_lag_24h            0
gas_price_lag_168h           0
coal_price_lag_24h           0
coal_price_lag_168h          0
co2_price_lag_24h            0
total_generation_lag_24h     0
total_generation_lag_168h    0
net_export_lag_24h           

Series([], dtype: int64)


=== TIME GAP CHECK ===
Missing hours: 0
DatetimeIndex([], dtype='datetime64[us]', freq='h')

=== ZERO COUNTS IN SUSPICIOUS COLUMNS ===
price_lag_24h                189
price_lag_168h               188
temperature                    3
load                           0
gas_price_lag_24h              0
gas_price_lag_168h             0
coal_price_lag_24h             0
coal_price_lag_168h            0
wind_speed                     0
co2_price_lag_24h              0
price_rolling_24h              0
price_rolling_168h             0
price_volatility_24h           0
total_generation_lag_24h       0
total_generation_lag_168h      0
net_export_lag_24h             0
net_export_lag_168h            0
dtype: int64

=== DESCRIPTIVE STATISTICS ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,is_peak_hour,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast
count,62544,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,...,62544.000,62544.000,6.254400e+04,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000
mean,2022-08-09 23:30:00,95.254,54279.022,2898.344,11901.620,6191.172,11.500,2.999,6.432,10.690,...,0.375,5514.424,3.618620e+08,0.420,2022.113,33287.885,-0.238,-0.357,14799.965,-12.509
min,2019-01-15 00:00:00,-500.000,30544.750,13.000,161.250,0.000,0.000,0.000,1.000,-10.763,...,0.000,0.000,0.000000e+00,0.000,2019.000,-14040.250,-8181.750,-13094.090,244.750,-44745.000
25%,2020-10-27 11:45:00,38.110,46783.250,1151.688,4585.000,0.000,5.750,1.000,3.000,4.825,...,0.000,0.000,0.000000e+00,0.000,2020.000,24705.688,-1474.000,-1051.312,6122.938,-5547.812
50%,2022-08-09 23:30:00,76.040,54153.875,2740.000,9166.125,221.750,11.500,3.000,6.000,10.225,...,0.000,0.000,1.174325e+07,0.000,2022.000,34230.375,-347.250,1.875,12019.695,-136.125
75%,2024-05-22 11:15:00,115.382,61848.625,4584.062,16905.798,9783.562,17.250,5.000,9.000,16.337,...,1.000,7613.812,5.729601e+08,1.000,2024.000,42606.500,1206.963,1201.812,21292.062,5487.000
max,2026-03-04 23:00:00,936.280,77585.750,7344.190,46617.250,50444.750,23.000,6.000,12.000,35.638,...,1.000,51550.500,3.067909e+09,1.000,2026.000,70214.250,9586.000,11920.250,51550.500,36628.250
std,NaN,92.985,9180.340,1887.114,9370.467,9604.880,6.922,2.000,3.479,7.497,...,0.484,9742.299,5.626834e+08,0.494,2.062,13282.082,2305.984,2845.453,10767.958,9905.366



=== HEAD ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast,date
0,2019-01-15 00:00:00,35.18,54057.25,4999.00,24868.25,0.0,0,1,1,1.2875,...,0.0,0.0,0,2019,24190.00,-1161.00,-1270.50,29867.25,-12156.0,2019-01-15
1,2019-01-15 01:00:00,35.82,52364.25,4923.25,25149.75,0.0,1,1,1,1.2000,...,0.0,0.0,0,2019,22291.25,-1693.00,205.75,30073.00,-11445.5,2019-01-15
2,2019-01-15 02:00:00,34.99,51553.50,4883.25,25871.00,0.0,2,1,1,1.2625,...,0.0,0.0,0,2019,20799.25,-810.75,681.25,30754.25,-10143.5,2019-01-15



=== TAIL ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast,date
62541,2026-03-04 21:00:00,151.88,58067.21,2345.98,5956.24,0.0,21,2,3,5.2625,...,8302.22,0.0,0,2026,49764.99,-3504.73,911.57,8302.22,1078.21,2026-03-04
62542,2026-03-04 22:00:00,142.24,54347.15,2438.99,6353.38,0.0,22,2,3,4.3625,...,0.00,0.0,0,2026,45554.78,-3720.06,490.15,8792.37,1954.08,2026-03-04
62543,2026-03-04 23:00:00,124.81,50585.48,2445.56,6559.53,0.0,23,2,3,3.8125,...,0.00,0.0,0,2026,41580.39,-3761.67,212.72,9005.09,2717.58,2026-03-04


In [84]:
df_check = pd.read_csv("../data/df_features.csv", parse_dates=["timestamp"])

final_feature_cols = [c for c in df_check.columns if c not in ["timestamp", "price"]]

print("Final number of model features:", len(final_feature_cols))
print(final_feature_cols)

Final number of model features: 36
['load', 'wind_offshore', 'wind_onshore', 'solar', 'hour', 'day_of_week', 'month', 'temperature', 'wind_speed', 'is_weekend', 'price_lag_24h', 'price_lag_168h', 'price_rolling_24h', 'price_rolling_168h', 'is_holiday', 'is_hol_or_week', 'price_volatility_24h', 'gas_price_lag_24h', 'gas_price_lag_168h', 'coal_price_lag_24h', 'coal_price_lag_168h', 'co2_price_lag_24h', 'total_generation_lag_24h', 'total_generation_lag_168h', 'net_export_lag_24h', 'net_export_lag_168h', 'is_peak_hour', 'wind_x_peak', 'solar_x_demand', 'is_crisis_period', 'year', 'residual_load', 'load_ramp', 'renewable_ramp', 'total_wind_forecast', 'delta_wind_forecast']


In [85]:
# =========================================================
# Final data quality check for df_features
# =========================================================

# Load the saved feature dataset
df_check = pd.read_csv("../data/df_features.csv", parse_dates=["timestamp"])

# Show basic shape and range
print("=== SHAPE ===")
print(df_check.shape)

print("\n=== DATE RANGE ===")
print(f"From: {df_check['timestamp'].min()}")
print(f"To:   {df_check['timestamp'].max()}")

# Check duplicate timestamps
print("\n=== DUPLICATE TIMESTAMPS ===")
print(df_check["timestamp"].duplicated().sum())

# Check missing values
print("\n=== MISSING VALUES PER COLUMN ===")
print(df_check.isnull().sum().sort_values(ascending=False))

print("\n=== TOTAL NUMBER OF MISSING VALUES ===")
print(df_check.isnull().sum().sum())

# Check daily completeness
print("\n=== ROWS PER DAY ===")
df_check["date"] = df_check["timestamp"].dt.date
day_counts = df_check.groupby("date").size()
print(day_counts.value_counts().sort_index())

print("\n=== INCOMPLETE OR OVERCOMPLETE DAYS ===")
display(day_counts[day_counts != 24])

# Check for time gaps
print("\n=== TIME GAP CHECK ===")
full_range = pd.date_range(
    start=df_check["timestamp"].min(),
    end=df_check["timestamp"].max(),
    freq="h"
)
missing_hours = full_range.difference(df_check["timestamp"])
print(f"Missing hours: {len(missing_hours)}")
print(missing_hours[:20])

# Columns where zero can be suspicious
suspicious_zero_cols = [
    "load", "temperature", "wind_speed",
    "gas_price_lag_24h", "gas_price_lag_168h",
    "coal_price_lag_24h", "coal_price_lag_168h",
    "co2_price_lag_24h",
    "price_lag_24h", "price_lag_168h",
    "price_rolling_24h", "price_rolling_168h",
    "price_volatility_24h",
    "total_generation_lag_24h", "total_generation_lag_168h",
    "net_export_lag_24h", "net_export_lag_168h"
]

existing_suspicious_cols = [c for c in suspicious_zero_cols if c in df_check.columns]

print("\n=== ZERO COUNTS IN SUSPICIOUS COLUMNS ===")
zero_counts = (df_check[existing_suspicious_cols] == 0).sum().sort_values(ascending=False)
print(zero_counts)

# Show descriptive statistics
print("\n=== DESCRIPTIVE STATISTICS ===")
display(df_check.describe().round(3))

# Preview
print("\n=== HEAD ===")
display(df_check.head(3))

print("\n=== TAIL ===")
display(df_check.tail(3))

=== SHAPE ===
(62544, 38)

=== DATE RANGE ===
From: 2019-01-15 00:00:00
To:   2026-03-04 23:00:00

=== DUPLICATE TIMESTAMPS ===
0

=== MISSING VALUES PER COLUMN ===
timestamp                    0
price                        0
load                         0
wind_offshore                0
wind_onshore                 0
solar                        0
hour                         0
day_of_week                  0
month                        0
temperature                  0
wind_speed                   0
is_weekend                   0
price_lag_24h                0
price_lag_168h               0
price_rolling_24h            0
price_rolling_168h           0
is_holiday                   0
is_hol_or_week               0
price_volatility_24h         0
gas_price_lag_24h            0
gas_price_lag_168h           0
coal_price_lag_24h           0
coal_price_lag_168h          0
co2_price_lag_24h            0
total_generation_lag_24h     0
total_generation_lag_168h    0
net_export_lag_24h           

Series([], dtype: int64)


=== TIME GAP CHECK ===
Missing hours: 0
DatetimeIndex([], dtype='datetime64[us]', freq='h')

=== ZERO COUNTS IN SUSPICIOUS COLUMNS ===
price_lag_24h                189
price_lag_168h               188
temperature                    3
load                           0
gas_price_lag_24h              0
gas_price_lag_168h             0
coal_price_lag_24h             0
coal_price_lag_168h            0
wind_speed                     0
co2_price_lag_24h              0
price_rolling_24h              0
price_rolling_168h             0
price_volatility_24h           0
total_generation_lag_24h       0
total_generation_lag_168h      0
net_export_lag_24h             0
net_export_lag_168h            0
dtype: int64

=== DESCRIPTIVE STATISTICS ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,is_peak_hour,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast
count,62544,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,...,62544.000,62544.000,6.254400e+04,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000
mean,2022-08-09 23:30:00,95.254,54279.022,2898.344,11901.620,6191.172,11.500,2.999,6.432,10.690,...,0.375,5514.424,3.618620e+08,0.420,2022.113,33287.885,-0.238,-0.357,14799.965,-12.509
min,2019-01-15 00:00:00,-500.000,30544.750,13.000,161.250,0.000,0.000,0.000,1.000,-10.763,...,0.000,0.000,0.000000e+00,0.000,2019.000,-14040.250,-8181.750,-13094.090,244.750,-44745.000
25%,2020-10-27 11:45:00,38.110,46783.250,1151.688,4585.000,0.000,5.750,1.000,3.000,4.825,...,0.000,0.000,0.000000e+00,0.000,2020.000,24705.688,-1474.000,-1051.312,6122.938,-5547.812
50%,2022-08-09 23:30:00,76.040,54153.875,2740.000,9166.125,221.750,11.500,3.000,6.000,10.225,...,0.000,0.000,1.174325e+07,0.000,2022.000,34230.375,-347.250,1.875,12019.695,-136.125
75%,2024-05-22 11:15:00,115.382,61848.625,4584.062,16905.798,9783.562,17.250,5.000,9.000,16.337,...,1.000,7613.812,5.729601e+08,1.000,2024.000,42606.500,1206.963,1201.812,21292.062,5487.000
max,2026-03-04 23:00:00,936.280,77585.750,7344.190,46617.250,50444.750,23.000,6.000,12.000,35.638,...,1.000,51550.500,3.067909e+09,1.000,2026.000,70214.250,9586.000,11920.250,51550.500,36628.250
std,NaN,92.985,9180.340,1887.114,9370.467,9604.880,6.922,2.000,3.479,7.497,...,0.484,9742.299,5.626834e+08,0.494,2.062,13282.082,2305.984,2845.453,10767.958,9905.366



=== HEAD ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast,date
0,2019-01-15 00:00:00,35.18,54057.25,4999.00,24868.25,0.0,0,1,1,1.2875,...,0.0,0.0,0,2019,24190.00,-1161.00,-1270.50,29867.25,-12156.0,2019-01-15
1,2019-01-15 01:00:00,35.82,52364.25,4923.25,25149.75,0.0,1,1,1,1.2000,...,0.0,0.0,0,2019,22291.25,-1693.00,205.75,30073.00,-11445.5,2019-01-15
2,2019-01-15 02:00:00,34.99,51553.50,4883.25,25871.00,0.0,2,1,1,1.2625,...,0.0,0.0,0,2019,20799.25,-810.75,681.25,30754.25,-10143.5,2019-01-15



=== TAIL ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast,date
62541,2026-03-04 21:00:00,151.88,58067.21,2345.98,5956.24,0.0,21,2,3,5.2625,...,8302.22,0.0,0,2026,49764.99,-3504.73,911.57,8302.22,1078.21,2026-03-04
62542,2026-03-04 22:00:00,142.24,54347.15,2438.99,6353.38,0.0,22,2,3,4.3625,...,0.00,0.0,0,2026,45554.78,-3720.06,490.15,8792.37,1954.08,2026-03-04
62543,2026-03-04 23:00:00,124.81,50585.48,2445.56,6559.53,0.0,23,2,3,3.8125,...,0.00,0.0,0,2026,41580.39,-3761.67,212.72,9005.09,2717.58,2026-03-04


In [86]:
# =========================================================
# Final data quality check for df_features
# =========================================================

# Load the saved feature dataset
df_check = pd.read_csv("../data/df_features.csv", parse_dates=["timestamp"])

# Show basic shape and range
print("=== SHAPE ===")
print(df_check.shape)

print("\n=== DATE RANGE ===")
print(f"From: {df_check['timestamp'].min()}")
print(f"To:   {df_check['timestamp'].max()}")

# Check duplicate timestamps
print("\n=== DUPLICATE TIMESTAMPS ===")
print(df_check["timestamp"].duplicated().sum())

# Check missing values
print("\n=== MISSING VALUES PER COLUMN ===")
print(df_check.isnull().sum().sort_values(ascending=False))

print("\n=== TOTAL NUMBER OF MISSING VALUES ===")
print(df_check.isnull().sum().sum())

# Check daily completeness
print("\n=== ROWS PER DAY ===")
df_check["date"] = df_check["timestamp"].dt.date
day_counts = df_check.groupby("date").size()
print(day_counts.value_counts().sort_index())

print("\n=== INCOMPLETE OR OVERCOMPLETE DAYS ===")
display(day_counts[day_counts != 24])

# Check for time gaps
print("\n=== TIME GAP CHECK ===")
full_range = pd.date_range(
    start=df_check["timestamp"].min(),
    end=df_check["timestamp"].max(),
    freq="h"
)
missing_hours = full_range.difference(df_check["timestamp"])
print(f"Missing hours: {len(missing_hours)}")
print(missing_hours[:20])

# Columns where zero can be suspicious
suspicious_zero_cols = [
    "load", "temperature", "wind_speed",
    "gas_price_lag_24h", "gas_price_lag_168h",
    "coal_price_lag_24h", "coal_price_lag_168h",
    "co2_price_lag_24h",
    "price_lag_24h", "price_lag_168h",
    "price_rolling_24h", "price_rolling_168h",
    "price_volatility_24h",
    "total_generation_lag_24h", "total_generation_lag_168h",
    "net_export_lag_24h", "net_export_lag_168h"
]

existing_suspicious_cols = [c for c in suspicious_zero_cols if c in df_check.columns]

print("\n=== ZERO COUNTS IN SUSPICIOUS COLUMNS ===")
zero_counts = (df_check[existing_suspicious_cols] == 0).sum().sort_values(ascending=False)
print(zero_counts)

# Show descriptive statistics
print("\n=== DESCRIPTIVE STATISTICS ===")
display(df_check.describe().round(3))

# Preview
print("\n=== HEAD ===")
display(df_check.head(3))

print("\n=== TAIL ===")
display(df_check.tail(3))

=== SHAPE ===
(62544, 38)

=== DATE RANGE ===
From: 2019-01-15 00:00:00
To:   2026-03-04 23:00:00

=== DUPLICATE TIMESTAMPS ===
0

=== MISSING VALUES PER COLUMN ===
timestamp                    0
price                        0
load                         0
wind_offshore                0
wind_onshore                 0
solar                        0
hour                         0
day_of_week                  0
month                        0
temperature                  0
wind_speed                   0
is_weekend                   0
price_lag_24h                0
price_lag_168h               0
price_rolling_24h            0
price_rolling_168h           0
is_holiday                   0
is_hol_or_week               0
price_volatility_24h         0
gas_price_lag_24h            0
gas_price_lag_168h           0
coal_price_lag_24h           0
coal_price_lag_168h          0
co2_price_lag_24h            0
total_generation_lag_24h     0
total_generation_lag_168h    0
net_export_lag_24h           

Series([], dtype: int64)


=== TIME GAP CHECK ===
Missing hours: 0
DatetimeIndex([], dtype='datetime64[us]', freq='h')

=== ZERO COUNTS IN SUSPICIOUS COLUMNS ===
price_lag_24h                189
price_lag_168h               188
temperature                    3
load                           0
gas_price_lag_24h              0
gas_price_lag_168h             0
coal_price_lag_24h             0
coal_price_lag_168h            0
wind_speed                     0
co2_price_lag_24h              0
price_rolling_24h              0
price_rolling_168h             0
price_volatility_24h           0
total_generation_lag_24h       0
total_generation_lag_168h      0
net_export_lag_24h             0
net_export_lag_168h            0
dtype: int64

=== DESCRIPTIVE STATISTICS ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,is_peak_hour,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast
count,62544,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,...,62544.000,62544.000,6.254400e+04,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000,62544.000
mean,2022-08-09 23:30:00,95.254,54279.022,2898.344,11901.620,6191.172,11.500,2.999,6.432,10.690,...,0.375,5514.424,3.618620e+08,0.420,2022.113,33287.885,-0.238,-0.357,14799.965,-12.509
min,2019-01-15 00:00:00,-500.000,30544.750,13.000,161.250,0.000,0.000,0.000,1.000,-10.763,...,0.000,0.000,0.000000e+00,0.000,2019.000,-14040.250,-8181.750,-13094.090,244.750,-44745.000
25%,2020-10-27 11:45:00,38.110,46783.250,1151.688,4585.000,0.000,5.750,1.000,3.000,4.825,...,0.000,0.000,0.000000e+00,0.000,2020.000,24705.688,-1474.000,-1051.312,6122.938,-5547.812
50%,2022-08-09 23:30:00,76.040,54153.875,2740.000,9166.125,221.750,11.500,3.000,6.000,10.225,...,0.000,0.000,1.174325e+07,0.000,2022.000,34230.375,-347.250,1.875,12019.695,-136.125
75%,2024-05-22 11:15:00,115.382,61848.625,4584.062,16905.798,9783.562,17.250,5.000,9.000,16.337,...,1.000,7613.812,5.729601e+08,1.000,2024.000,42606.500,1206.963,1201.812,21292.062,5487.000
max,2026-03-04 23:00:00,936.280,77585.750,7344.190,46617.250,50444.750,23.000,6.000,12.000,35.638,...,1.000,51550.500,3.067909e+09,1.000,2026.000,70214.250,9586.000,11920.250,51550.500,36628.250
std,NaN,92.985,9180.340,1887.114,9370.467,9604.880,6.922,2.000,3.479,7.497,...,0.484,9742.299,5.626834e+08,0.494,2.062,13282.082,2305.984,2845.453,10767.958,9905.366



=== HEAD ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast,date
0,2019-01-15 00:00:00,35.18,54057.25,4999.00,24868.25,0.0,0,1,1,1.2875,...,0.0,0.0,0,2019,24190.00,-1161.00,-1270.50,29867.25,-12156.0,2019-01-15
1,2019-01-15 01:00:00,35.82,52364.25,4923.25,25149.75,0.0,1,1,1,1.2000,...,0.0,0.0,0,2019,22291.25,-1693.00,205.75,30073.00,-11445.5,2019-01-15
2,2019-01-15 02:00:00,34.99,51553.50,4883.25,25871.00,0.0,2,1,1,1.2625,...,0.0,0.0,0,2019,20799.25,-810.75,681.25,30754.25,-10143.5,2019-01-15



=== TAIL ===


,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,wind_x_peak,solar_x_demand,is_crisis_period,year,residual_load,load_ramp,renewable_ramp,total_wind_forecast,delta_wind_forecast,date
62541,2026-03-04 21:00:00,151.88,58067.21,2345.98,5956.24,0.0,21,2,3,5.2625,...,8302.22,0.0,0,2026,49764.99,-3504.73,911.57,8302.22,1078.21,2026-03-04
62542,2026-03-04 22:00:00,142.24,54347.15,2438.99,6353.38,0.0,22,2,3,4.3625,...,0.00,0.0,0,2026,45554.78,-3720.06,490.15,8792.37,1954.08,2026-03-04
62543,2026-03-04 23:00:00,124.81,50585.48,2445.56,6559.53,0.0,23,2,3,3.8125,...,0.00,0.0,0,2026,41580.39,-3761.67,212.72,9005.09,2717.58,2026-03-04


In [87]:
df_check = pd.read_csv("../data/df_features.csv", parse_dates=["timestamp"])

final_feature_cols = [c for c in df_check.columns if c not in ["timestamp", "price"]]

print("Final number of model features:", len(final_feature_cols))
print(final_feature_cols)

Final number of model features: 36
['load', 'wind_offshore', 'wind_onshore', 'solar', 'hour', 'day_of_week', 'month', 'temperature', 'wind_speed', 'is_weekend', 'price_lag_24h', 'price_lag_168h', 'price_rolling_24h', 'price_rolling_168h', 'is_holiday', 'is_hol_or_week', 'price_volatility_24h', 'gas_price_lag_24h', 'gas_price_lag_168h', 'coal_price_lag_24h', 'coal_price_lag_168h', 'co2_price_lag_24h', 'total_generation_lag_24h', 'total_generation_lag_168h', 'net_export_lag_24h', 'net_export_lag_168h', 'is_peak_hour', 'wind_x_peak', 'solar_x_demand', 'is_crisis_period', 'year', 'residual_load', 'load_ramp', 'renewable_ramp', 'total_wind_forecast', 'delta_wind_forecast']


## Final dataset features and leakage check

Description of the final dataset columns and whether there are any leakage concerns.

### 0. Metadata and target

1. **`timestamp`**  
   Hourly timestamp of the observation.  
   Not a model feature itself, but used to preserve chronological order and build time-based features.

2. **`price`**  
   Day-ahead electricity price for the target hour.  
   This is the prediction target, not an input feature.

3. **`date`**  
   Calendar date derived from `timestamp`.  
   This was added temporarily in the check step to inspect rows per day. It is not part of the final model input.

---

### 1. Forecasted market and system inputs

4. **`load`**  
   Forecasted electricity load for that hour.  

5. **`wind_offshore`**  
   Forecasted offshore wind generation for that hour.  

6. **`wind_onshore`**  
   Forecasted onshore wind generation for that hour.  

7. **`solar`**  
   Forecasted solar generation for that hour.  

8. **`temperature`**  
   Weather temperature feature merged by timestamp.  
   **Leakage note:** acceptable in the setup as an external explanatory variable.

9. **`wind_speed`**  
   Weather wind speed feature merged by timestamp.  
   **Leakage note:** acceptable in the same sense as `temperature`.

---

### 2. Calendar features

10. **`hour`**  
    Hour of the day from the timestamp.  

11. **`day_of_week`**  
    Day of the week from the timestamp.  

12. **`month`**  
    Month from the timestamp.  

13. **`year`**  
    Year from the timestamp.  

14. **`is_weekend`**  
    Indicator for Saturday or Sunday.  

15. **`is_holiday`**  
    Indicator for German public holidays.  

16. **`is_hol_or_week`**  
    Indicator for holiday or weekend.  

17. **`is_crisis_period`**  
    Indicator for the 2021–2023 energy crisis period.  

18. **`is_peak_hour`**  
    Indicator for morning and evening peak hours.  

---

### 3. Safe target-history features

19. **`price_lag_24h`**  
    Electricity price from 24 hours earlier.  

20. **`price_lag_168h`**  
    Electricity price from the same hour one week earlier.  

21. **`price_rolling_24h`**  
    Rolling mean of past prices over the previous 24 hours.  
    **Leakage note:** this feature would be leaky if it included the current hour price. It is safe here because it was redefined with `shift(1)` before rolling, so only past prices are used.

22. **`price_rolling_168h`**  
    Rolling mean of past prices over the previous 168 hours.  
    **Leakage note:** this feature would be leaky if it included the current hour price. It is safe here because it was redefined with `shift(1)` before rolling, so only past prices are used.

23. **`price_volatility_24h`**  
    Rolling standard deviation of past prices over the previous 24 hours.  
    **Leakage note:** this feature would be leaky if it included the current hour price. It is safe here because it was redefined with `shift(1)` before rolling, so only past prices are used.

---

### 4. Lagged fuel-price features

24. **`gas_price_lag_24h`**  
    Gas price from 24 hours earlier.  

25. **`gas_price_lag_168h`**  
    Gas price from 168 hours earlier.  

26. **`coal_price_lag_24h`**  
    Coal price from 24 hours earlier.  

27. **`coal_price_lag_168h`**  
    Coal price from 168 hours earlier.  

28. **`co2_price_lag_24h`**  
    CO₂ price from 24 hours earlier.  

---

### 5. Lagged past system-state features

29. **`total_generation_lag_24h`**  
    Total realized generation from 24 hours earlier.  
    **Leakage note:** same-hour `total_generation` would be leaky, but the lagged version is safe because it uses past known values.

30. **`total_generation_lag_168h`**  
    Total realized generation from 168 hours earlier.  
    **Leakage note:** safe for the same reason as above.

31. **`net_export_lag_24h`**  
    Net export from 24 hours earlier.  
    **Leakage note:** same-hour `net_export` would be leaky, but the lagged version is safe because it uses past known values.

32. **`net_export_lag_168h`**  
    Net export from 168 hours earlier.  
    **Leakage note:** safe for the same reason as above.

---

### 6. Interaction features

33. **`wind_x_peak`**  
    Forecasted total wind during peak hours.  
    Built from forecasted wind and the peak-hour flag.

34. **`solar_x_demand`**  
    Forecasted solar multiplied by forecasted load.  
    Built only from forecast-based inputs.

---

### 7. Additional forecast-based engineered features

35. **`residual_load`**  
    Forecasted load minus forecasted renewables (`wind_onshore + wind_offshore + solar`).  
    **Leakage note:** safe because all source columns are forecast-based.

36. **`load_ramp`**  
    Hour-to-hour change in forecasted load.  
    **Leakage note:** safe because it uses forecasted load, not actual load.

37. **`renewable_ramp`**  
    Hour-to-hour change in forecasted renewable generation.  
    **Leakage note:** safe because it is built from forecasted renewable inputs.

38. **`total_wind_forecast`**  
    Total forecasted wind generation (`wind_onshore + wind_offshore`).  

39. **`delta_wind_forecast`**  
    Difference between the current total wind forecast and the total wind forecast 24 hours earlier.  
    **Leakage note:** safe because it is based on forecast values and past comparison.

---

## Excluded because of leakage risk or conservative filtering

These are not part of the final strict model feature set:

- **`gas_price`**: same-hour gas price  
- **`coal_price`**: same-hour coal price  
- **`co2_price`**: same-hour CO₂ price  
- **`gas_x_peak`**: depends on same-hour gas price  
- **`is_high_price_regime`**: built directly from the target price  
- **`renewable_share`**: depends on same-hour realized `total_generation`  
- **`renewable_share_x_peak`**: built from `renewable_share`  
- **`dispatchable_gen`**: uses same-hour realized coal/gas/nuclear generation  
- **`demand_supply_gap`**: uses same-hour realized total generation  
- **`actual_load`**: realized load of the target hour  
- **`actual_wind_offshore`**: realized offshore wind generation  
- **`actual_wind_onshore`**: realized onshore wind generation  
- **`actual_solar`**: realized solar generation  
- **`total_generation`**: same-hour realized total generation  
- **`coal_generation`**: same-hour realized coal generation  
- **`gas_generation`**: same-hour realized gas generation  
- **`nuclear_generation`**: same-hour realized nuclear generation  
- **`net_export`**: same-hour realized cross-border physical flow  

---

## Overall conclusion

The final saved modeling dataset contains:

- **36 strict model features**
- plus **`timestamp`**
- plus **`price`**

In the check preview, a temporary **`date`** column was added, which is why you saw **39 columns** there.

The features with the main leakage concerns were the rolling price features and any same-hour realized system-state variables. In the final version:

- the rolling price features were corrected with `shift(1)`
- the same-hour realized variables were excluded
- only lagged versions of realized system-state variables were kept